In [1]:
#Imports

import pandas as pd
import numpy as np

import mlflow
import mlflow.sklearn

from sklearn.model_selection import (

    train_test_split,

    cross_val_score

)

from sklearn.metrics import (

    accuracy_score

)

from xgboost import (

    XGBClassifier

)

In [2]:
#Configurações do MLflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")

mlflow.set_experiment("XGboost")

mlflow.sklearn.autolog()

In [3]:
#CArregar o dataset

df = pd.read_csv("../DATASET/dataset_expandido.csv")

In [4]:
#Preparar os dados

df["YearsCodePro_Num"]=(

    df["YearsCodePro"]

)

df["YearsCodePro_Num"]=(
    df["YearsCodePro_Num"]

    .replace({

        "Less than 1 year":0,

        "More than 50 years":51,

        "Sem dado":0

    })
)

df["YearsCodePro_Num"]=pd.to_numeric(

    df["YearsCodePro_Num"]

)

features=[

    "YearsCodePro_Num",

    "WorkExp",

    "Age_Code"

]

target="JobSat_Class"

X=df[features]

y=df[target]

y=y.replace({

    "Baixo":0,

    "Medio":1,

    "Alto":2

})

C:\Users\luisr\AppData\Local\Temp\ipykernel_24392\1816381890.py:45: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y=y.replace({


In [5]:
#Split dos dados

X_train,X_test,y_train,y_test=(

    train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )
)

In [6]:
#Função do XGBoost

def run_xgb_experiment(

    run_name,

    estimators,

    lr,

    depth

):

    with mlflow.start_run(

        run_name=run_name

    ):

        model=XGBClassifier(

            n_estimators=
            estimators,

            learning_rate=
            lr,

            max_depth=
            depth,

            random_state=42,

            eval_metric=
            "mlogloss"

        )

        model.fit(

            X_train,

            y_train

        )

        y_pred=model.predict(

            X_test

        )

        accuracy=accuracy_score(

            y_test,

            y_pred

        )

        cv=cross_val_score(

            model,

            X_train,

            y_train,

            cv=5

        )

        mlflow.log_metric(

            "accuracy",

            accuracy

        )

        mlflow.log_metric(

            "cv_mean",

            cv.mean()

        )

        print(run_name)

        print(
            accuracy
        )

In [9]:
#Experiencia 0 - Baseline

run_xgb_experiment(

    "Experiment_0_Baseline",
    100,
    0.1,
    3
)

2026/05/24 18:03:55 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



Experiment_0_Baseline
0.7495466634429401


In [10]:
#Experiencia 1 -

run_xgb_experiment(

    "Experiment_1_200",
    200,
    0.1,
    3
)

Experiment_1_200
0.7504231141199227


In [11]:
#Experiencia 2 

run_xgb_experiment(
    "Experiment_2_lr005",
    100,
    0.05,
    3
)

Experiment_2_lr005
0.7482168762088974


In [12]:
#Experiencia 3

run_xgb_experiment(

    "Experiment_3_lr02",
    100,
    0.2,
    3
)

Experiment_3_lr02
0.7505137814313346


In [13]:
#Experiencia 4

run_xgb_experiment(

    "Experiment_4_depth5",
    100,
    0.1,
    5
)

Experiment_4_depth5
0.7534755802707931


In [14]:
#Experiencia 5

features_extra=[
    "YearsCodePro_Num",
    "WorkExp",
    "Age_Code",
    "JobSatPoints_1",
    "JobSatPoints_4",
    "JobSatPoints_5"
]

X=df[features_extra]

X_train,X_test,y_train,y_test=(

    train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )
)

run_xgb_experiment(
    "Experiment_5_Features",
    100,
    0.1,
    3
)

Experiment_5_Features
0.757948500967118


# Análise de Resultados


O XGBoost foi um dos modelos com melhor desempenho ao longo do projeto, terminando entre os algoritmos mais competitivos da classificação final. Este resultado era esperado, uma vez que o XGBoost é uma evolução dos métodos tradicionais de Gradient Boosting, incorporando mecanismos adicionais de otimização e regularização que permitem melhorar tanto a precisão como a capacidade de generalização do modelo.

Durante as experiências realizadas foram testadas diferentes configurações dos principais hiperparâmetros, procurando encontrar o melhor equilíbrio entre desempenho e complexidade. Tal como acontece noutros algoritmos de boosting, pequenas alterações nos parâmetros podem provocar diferenças significativas nos resultados, o que torna a fase de otimização particularmente importante.

Os resultados obtidos demonstraram que o XGBoost conseguiu atingir valores muito competitivos de Accuracy, posicionando-se próximo dos melhores modelos do projeto. Este desempenho confirma a capacidade do algoritmo para identificar padrões complexos nos dados e produzir previsões bastante precisas, mesmo quando comparado com outros métodos ensemble já bastante robustos.

Uma das principais vantagens observadas foi a utilização de mecanismos de regularização, que ajudam a reduzir o risco de overfitting. Além disso, o algoritmo utiliza estratégias de otimização que tornam o processo de aprendizagem mais eficiente e permitem obter modelos mais estáveis do que os obtidos com abordagens tradicionais de boosting.

Apesar de não ter alcançado a primeira posição do ranking final, o XGBoost demonstrou um desempenho bastante sólido e confirmou a reputação que possui na área de Machine Learning. Os resultados obtidos mostram que este algoritmo é uma excelente escolha para problemas de classificação, conseguindo combinar elevada capacidade preditiva com uma boa capacidade de generalização dos dados.

# Hyperparameter Optimization - XGBoost

In [7]:
import pandas as pd
import numpy as np

import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score

from xgboost import XGBRegressor

import optuna

In [8]:
target = "ConvertedCompYearly"

features = [
    "YearsCodePro_Num",
    "WorkExp",
    "Age_Code"
]

X = df[features]

y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [9]:
mlflow.sklearn.autolog(
    disable=True
)

In [10]:
param_grid = {

    "n_estimators":[
        50,
        100,
        200
    ],

    "learning_rate":[
        0.01,
        0.1,
        0.2
    ],

    "max_depth":[
        3,
        5,
        7
    ]

}

grid = GridSearchCV(

    estimator=XGBRegressor(
        random_state=42
    ),

    param_grid=param_grid,

    cv=10,

    scoring="r2",

    n_jobs=-1

)

grid.fit(

    X_train,

    y_train

)

print(

    "Best Parameters:",

    grid.best_params_

)

print(

    "Best Score:",

    grid.best_score_

)

Best Parameters: {'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 200}
Best Score: 0.11616773370463082


In [11]:
with mlflow.start_run(

    run_name="GridSearch_XGBoost"

):

    mlflow.log_param(

        "method",

        "GridSearch"

    )

    mlflow.log_params(

        grid.best_params_

    )

    mlflow.log_metric(

        "best_score",

        grid.best_score_

    )

2026/05/29 19:23:25 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



In [12]:
def objective(

    trial

):

    n_estimators = trial.suggest_categorical(

        "n_estimators",

        [

            50,

            100,

            200

        ]

    )

    learning_rate = trial.suggest_categorical(

        "learning_rate",

        [

            0.01,

            0.1,

            0.2

        ]

    )

    max_depth = trial.suggest_categorical(

        "max_depth",

        [

            3,

            5,

            7

        ]

    )

    model = XGBRegressor(

        n_estimators=n_estimators,

        learning_rate=learning_rate,

        max_depth=max_depth,

        random_state=42,

        n_jobs=-1

    )

    scores = cross_val_score(

        model,

        X_train,

        y_train,

        cv=10,

        scoring="r2",

        n_jobs=-1

    )

    return scores.mean()

In [13]:
study = optuna.create_study(

    direction="maximize"

)

study.optimize(

    objective,

    n_trials=100

)

print(

    "Best Parameters:",

    study.best_params

)

print(

    "Best Score:",

    study.best_value

)

[I 2026-05-29 19:23:33,624] A new study created in memory with name: no-name-9bb04259-739b-4fdd-8392-52603466357a
[I 2026-05-29 19:23:39,176] Trial 0 finished with value: 0.08778701692880998 and parameters: {'n_estimators': 200, 'learning_rate': 0.01, 'max_depth': 7}. Best is trial 0 with value: 0.08778701692880998.
[I 2026-05-29 19:23:40,334] Trial 1 finished with value: 0.1046630771385975 and parameters: {'n_estimators': 50, 'learning_rate': 0.2, 'max_depth': 5}. Best is trial 1 with value: 0.1046630771385975.
[I 2026-05-29 19:23:43,773] Trial 2 finished with value: 0.09541373486015817 and parameters: {'n_estimators': 100, 'learning_rate': 0.01, 'max_depth': 5}. Best is trial 1 with value: 0.1046630771385975.
[I 2026-05-29 19:23:45,196] Trial 3 finished with value: 0.01901048103030274 and parameters: {'n_estimators': 50, 'learning_rate': 0.01, 'max_depth': 3}. Best is trial 1 with value: 0.1046630771385975.
[I 2026-05-29 19:23:46,366] Trial 4 finished with value: 0.01901048103030274 

Best Parameters: {'n_estimators': 200, 'learning_rate': 0.01, 'max_depth': 5}
Best Score: 0.11616773370463082


In [14]:
with mlflow.start_run(

    run_name="Optuna_XGBoost"

):

    mlflow.log_param(

        "method",

        "Optuna"

    )

    mlflow.log_params(

        study.best_params

    )

    mlflow.log_metric(

        "best_score",

        study.best_value

    )